# StatsBomb FIFA Women's passing network example: NZL 1-0 NOR

In [2]:
import pandas as pd
import numpy as np

## Load the events data and build the dataframe

In [10]:
MATCH_ID = 3893787
df = pd.read_json(f'/Users/shawnhan/Desktop/Pioneer/Pioneer_Final_Project/Download_Statsbomb/wwc2023/events/{MATCH_ID}.json')

# Filter rows where the 'name' value inside the dict contains 'Pass' or 'Ball Receipt'
filtered_df = df[df['type'].apply(
    lambda x: (
        isinstance(x, dict) and (
            'pass' in x.get('name', '').lower() or
            'ball receipt' in x.get('name', '').lower()
        )
    )
)]

# Extract additional information
def extract_pass_success(row):
    if 'pass' in row and isinstance(row['pass'], dict):
        outcome = row['pass'].get('outcome')
        if outcome and isinstance(outcome, dict):
            return outcome.get('name', 'Unknown')
    return 'Successful'  # If no outcome specified, assume successful

def extract_receipt_success(row):
    if 'ball_receipt' in row and isinstance(row['ball_receipt'], dict):
        outcome = row['ball_receipt'].get('outcome')
        if outcome and isinstance(outcome, dict):
            return outcome.get('name', 'Unknown')
    return 'Successful'  # If no outcome specified, assume successful

def extract_pressure_status(row):
    return row.get('under_pressure', False)

def extract_event_outcome(row):
    event_type = row['type'].get('name', '') if isinstance(row['type'], dict) else ''
    
    if 'pass' in event_type.lower():
        return extract_pass_success(row)
    elif 'ball receipt' in event_type.lower():
        return extract_receipt_success(row)
    else:
        return 'Unknown'

def extract_pass_type(row):
    if 'pass' in row and isinstance(row['pass'], dict):
        pass_type = row['pass'].get('type')
        if pass_type and isinstance(pass_type, dict):
            return pass_type.get('name')
    return None

# Create the subset with additional columns
subset_df = filtered_df[['minute', 'second', 'type', 'team', 'player', 'location']].copy()

# Add new columns
subset_df['under_pressure'] = filtered_df.apply(extract_pressure_status, axis=1)
subset_df['outcome'] = filtered_df.apply(extract_event_outcome, axis=1)
subset_df['pass_type'] = filtered_df.apply(extract_pass_type, axis=1)

# Filter for only successful passes and receipts
successful_df = subset_df[subset_df['outcome'] == 'Successful'].copy()

# Select only the specified columns
final_df = successful_df[['minute', 'second', 'type', 'team', 'player', 'location', 'under_pressure', 'pass_type']]

print("Original dataset shape:", subset_df.shape)
print("Successful events shape:", final_df.shape)
print(f"Filtered out {subset_df.shape[0] - final_df.shape[0]} unsuccessful events")

final_df.head(10)

Original dataset shape: (1583, 9)
Successful events shape: (1076, 8)
Filtered out 507 unsuccessful events


,minute,second,type,team,player,location,under_pressure,pass_type
4,0,0,"{'id': 30, 'name': 'Pass'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 392266, 'name': 'Jacqueline Anne Hand'}","[60.0, 40.0]",NaN,Kick Off
5,0,0,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 401642, 'name': 'Malia Grace Steinmetz'}","[53.9, 36.1]",NaN,None
7,0,1,"{'id': 30, 'name': 'Pass'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 401642, 'name': 'Malia Grace Steinmetz'}","[52.8, 37.1]",NaN,None
8,0,3,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 25662, 'name': 'Catherine Joan Bott'}","[34.1, 62.5]",NaN,None
12,0,8,"{'id': 30, 'name': 'Pass'}","{'id': 852, 'name': 'Norway Women's'}","{'id': 401645, 'name': 'Mathilde Hauge Harviken'}","[31.2, 17.2]",NaN,Recovery
13,0,9,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 852, 'name': 'Norway Women's'}","{'id': 191851, 'name': 'Julie Blakstad'}","[46.5, 11.2]",NaN,None
19,0,23,"{'id': 30, 'name': 'Pass'}","{'id': 852, 'name': 'Norway Women's'}","{'id': 276288, 'name': 'Tuva Hansen'}","[44.0, 0.1]",NaN,Throw-in
21,0,24,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 852, 'name': 'Norway Women's'}","{'id': 191851, 'name': 'Julie Blakstad'}","[47.7, 10.7]",1.0,None
27,0,27,"{'id': 30, 'name': 'Pass'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 66217, 'name': 'Indiah-Paige Janita Ril...","[82.6, 75.1]",1.0,None
28,0,28,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 392266, 'name': 'Jacqueline Anne Hand'}","[96.3, 71.7]",NaN,None


## Load the xT grid

In [14]:
xt_grid = pd.read_csv('/Users/shawnhan/Desktop/Pioneer/Pioneer_Final_Project/wwc2023_trained_xT_grid.csv', header=None).values

xT_rows, xT_cols = xt_grid.shape

def calculate_xt(actions_df):
    df = actions_df.copy()

    # Standard pitch dimensions used by socceraction/xT models are 105x68. We scale the StatsBomb 120x80 coordinates proportionally to this standard.
    
    df['x_start'] = df['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
    df['y_start'] = df['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)
    df['x_end'] = df['pass_end_location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
    df['y_end'] = df['pass_end_location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)

    df['x_start_scaled'] = df['x_start'] * (105/120)
    df['y_start_scaled'] = df['y_start'] * (68/80)
    df['x_end_scaled'] = df['x_end'] * (105/120)
    df['y_end_scaled'] = df['y_end'] * (68/80)
    
    # Bin the coordinates into the 16x12 grid zones
    df['x_start_bin'] = pd.cut(df['x_start_scaled'], bins=xT_cols, labels=False, include_lowest=True)
    df['y_start_bin'] = pd.cut(df['y_start_scaled'], bins=xT_rows, labels=False, include_lowest=True)
    df['x_end_bin'] = pd.cut(df['x_end_scaled'], bins=xT_cols, labels=False, include_lowest=True)
    df['y_end_bin'] = pd.cut(df['y_end_scaled'], bins=xT_rows, labels=False, include_lowest=True)

    def lookup_xt_value(x_bin, y_bin):
        if pd.notna(x_bin) and pd.notna(y_bin) and 0 <= int(y_bin) < xT_rows and 0 <= int(x_bin) < xT_cols:
            return xt_grid[int(y_bin), int(x_bin)]
        return 0
    
    df['xt_start'] = df.apply(lambda row: lookup_xt_value(row['x_start_bin'], row['y_start_bin']), axis=1)
    df['xt_end'] = df.apply(lambda row: lookup_xt_value(row['x_end_bin'], row['y_end_bin']), axis=1)
    
    # Calculate the change in xT, which is the value of the action
    df['xt_added'] = df['xt_end'] - df['xt_start']
    
    return df

# Use it on final_df which we've been filtered
final_df_with_xt = calculate_xt_for_passes(final_df)

final_df_with_xt.head(20)

,minute,second,type,team,player,location,under_pressure,pass_type,x_coord,y_coord,x_scaled,y_scaled,x_bin,y_bin,xT_value
4,0,0,"{'id': 30, 'name': 'Pass'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 392266, 'name': 'Jacqueline Anne Hand'}","[60.0, 40.0]",NaN,Kick Off,60.0,40.0,50.000000,50.000,7,6,0.006072
5,0,0,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 401642, 'name': 'Malia Grace Steinmetz'}","[53.9, 36.1]",NaN,None,53.9,36.1,44.916667,45.125,6,5,0.005501
7,0,1,"{'id': 30, 'name': 'Pass'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 401642, 'name': 'Malia Grace Steinmetz'}","[52.8, 37.1]",NaN,None,52.8,37.1,44.000000,46.375,6,6,0.005340
8,0,3,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 25662, 'name': 'Catherine Joan Bott'}","[34.1, 62.5]",NaN,None,34.1,62.5,28.416667,78.125,4,10,0.003701
12,0,8,"{'id': 30, 'name': 'Pass'}","{'id': 852, 'name': 'Norway Women's'}","{'id': 401645, 'name': 'Mathilde Hauge Harviken'}","[31.2, 17.2]",NaN,Recovery,31.2,17.2,26.000000,21.500,3,2,0.002650
13,0,9,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 852, 'name': 'Norway Women's'}","{'id': 191851, 'name': 'Julie Blakstad'}","[46.5, 11.2]",NaN,None,46.5,11.2,38.750000,14.000,5,1,0.003499
19,0,23,"{'id': 30, 'name': 'Pass'}","{'id': 852, 'name': 'Norway Women's'}","{'id': 276288, 'name': 'Tuva Hansen'}","[44.0, 0.1]",NaN,Throw-in,44.0,0.1,36.666667,0.125,5,0,5.000000
21,0,24,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 852, 'name': 'Norway Women's'}","{'id': 191851, 'name': 'Julie Blakstad'}","[47.7, 10.7]",1.0,None,47.7,10.7,39.750000,13.375,6,1,0.004296
27,0,27,"{'id': 30, 'name': 'Pass'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 66217, 'name': 'Indiah-Paige Janita Ril...","[82.6, 75.1]",1.0,None,82.6,75.1,68.833333,93.875,10,12,0.009333
28,0,28,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1215, 'name': 'New Zealand Women's'}","{'id': 392266, 'name': 'Jacqueline Anne Hand'}","[96.3, 71.7]",NaN,None,96.3,71.7,80.250000,89.625,12,11,0.016488


## Weighting

In [16]:
def build_weighted_passing_network(final_df_with_xt):
    
    df = final_df_with_xt.copy()
    df['player_name'] = df['player'].apply(lambda x: x.get('name', '') if isinstance(x, dict) else '')
    df['team_name'] = df['team'].apply(lambda x: x.get('name', '') if isinstance(x, dict) else '')
    df['event_type'] = df['type'].apply(lambda x: x.get('name', '') if isinstance(x, dict) else '')
    
    df = df.sort_values(['minute', 'second']).reset_index(drop=True)
    
    pass_connections = []
    
    for team_name in df['team_name'].unique():
        team_data = df[df['team_name'] == team_name].reset_index(drop=True)
        
        i = 0
        while i < len(team_data) - 1:
            current_event = team_data.iloc[i]
            
            if current_event['event_type'] == 'Pass':
                j = i + 1
                while j < len(team_data):
                    next_event = team_data.iloc[j]
                    
                    if next_event['event_type'] == 'Ball Receipt*':
                        passer = current_event['player_name']
                        receiver = next_event['player_name']
                        
                        if passer != receiver and passer != '' and receiver != '':
                            start_xt = current_event['xT_value']  
                            end_xt = next_event['xT_value']      
                            xt_diff = end_xt - start_xt
                            
                            passer_under_pressure = current_event.get('under_pressure', False)
                            receiver_under_pressure = next_event.get('under_pressure', False)
                            
                            if pd.isna(passer_under_pressure):
                                passer_under_pressure = False
                            if pd.isna(receiver_under_pressure):
                                receiver_under_pressure = False
                            
                            if xt_diff > 0:
                                base_weight = 1 + (xt_diff * 10)  # Threat-increasing
                            else:
                                base_weight = max(0.1, 1 + (xt_diff * 5))  # Threat-decreasing
                            
                            # Pressure bonuses
                            pressure_bonus = 0
                            
                            if passer_under_pressure:
                                pressure_bonus += 0.15  

                            if receiver_under_pressure:
                                pressure_bonus += 0.15  
                            
                            if passer_under_pressure and receiver_under_pressure:
                                pressure_bonus += 0.2  
                            
                            final_weight = base_weight + pressure_bonus
                            
                            pass_connections.append({
                                'passer': passer,
                                'receiver': receiver,
                                'team': team_name,
                                'minute': current_event['minute'],
                                'second': current_event['second'],
                                'start_xt': start_xt,
                                'end_xt': end_xt,
                                'xt_diff': xt_diff,
                                'passer_under_pressure': passer_under_pressure,      
                                'receiver_under_pressure': receiver_under_pressure,  
                                'pressure_bonus': pressure_bonus,                    
                                'base_weight': base_weight,                          
                                'final_weight': final_weight                         
                            })
                        break
                    elif next_event['event_type'] == 'Pass':
                        break
                    j += 1
            i += 1
    
    passes_df = pd.DataFrame(pass_connections)
    
    if len(passes_df) == 0:
        return None, None, None
    
    # Build weighted adjacency matrix
    all_players = sorted(list(set(passes_df['passer'].tolist() + passes_df['receiver'].tolist())))
    n_players = len(all_players)
    player_to_idx = {player: idx for idx, player in enumerate(all_players)}
    
    # Use final weights (including pressure bonuses)
    adjacency_matrix = np.zeros((n_players, n_players))
    
    for _, row in passes_df.iterrows():
        passer_idx = player_to_idx[row['passer']]
        receiver_idx = player_to_idx[row['receiver']]
        adjacency_matrix[passer_idx, receiver_idx] += row['final_weight']  
    
    return adjacency_matrix, all_players, passes_df


## Get the scores

In [17]:
def calculate_weighted_pagerank(adjacency_matrix, players, damping_factor=0.85, max_iterations=100, tolerance=1e-6):
    n = len(players)
    
    transition_matrix = adjacency_matrix.copy()
    
    # Normalize
    row_sums = transition_matrix.sum(axis=1)
    row_sums[row_sums == 0] = 1
    transition_matrix = transition_matrix / row_sums[:, np.newaxis]
    
    # Initialize
    scores = np.ones(n) / n
    
    # Iteration
    for iteration in range(max_iterations):
        new_scores = (1 - damping_factor) / n + damping_factor * transition_matrix.T.dot(scores)
        
        # Convergence
        if np.abs(new_scores - scores).sum() < tolerance:
            print(f"Converged after {iteration + 1} iterations")
            break
        
        scores = new_scores
    
    return scores

def normalize_wpr_scores(results_df): # For cross match comparison
    
    scores = results_df['WPR_score'].copy()
    
    min_score = scores.min()
    max_score = scores.max()
    
    if max_score > min_score: 
        normalized_scores = ((scores - min_score) / (max_score - min_score)) * 100
    else:
        normalized_scores = scores * 100  
    
    results_df['WPR_score_normalized'] = normalized_scores.round(2)
    return results_df

weighted_adj, players, weighted_passes = build_weighted_passing_network(final_df_with_xt)
scores = calculate_weighted_pagerank(weighted_adj, players)

player_info = {}
team_info = {}
team_id_info = {}

for _, row in final_df_with_xt.iterrows():
    player_dict = row['player']
    team_dict = row['team']
    
    if isinstance(player_dict, dict) and isinstance(team_dict, dict):
        player_name = player_dict.get('name', '')
        player_id = player_dict.get('id', '')
        team_name = team_dict.get('name', '')
        team_id = team_dict.get('id', '')
        
        if player_name:
            player_info[player_name] = player_id
            team_info[player_name] = team_name
            team_id_info[player_name] = team_id

results_df = pd.DataFrame({ 
    'player': players,
    'player_id': [player_info.get(p, 'Unknown') for p in players],
    'team': [team_info.get(p, 'Unknown') for p in players],
    'team_id': [team_id_info.get(p, 'Unknown') for p in players],
    'WPR_score': scores
})

results_df = results_df.sort_values('WPR_score', ascending=False).reset_index(drop=True)
results_df['match_rank'] = range(1, len(results_df) + 1)
results_df['team_rank'] = results_df.groupby('team')['WPR_score'].rank(ascending=False, method='dense').astype(int)

results_df = normalize_wpr_scores(results_df)
results_df.head()

Converged after 19 iterations


,player,player_id,team,team_id,WPR_score,match_rank,team_rank,WPR_score_normalized
0,Alexandra Riley,21062,New Zealand Women's,1215,0.112032,1,1,100.00
1,Tuva Hansen,276288,Norway Women's,852,0.095016,2,1,84.01
2,Hannah Wilkinson,25658,New Zealand Women's,1215,0.079922,3,2,69.83
3,Elizabeth Doon Hassett,25663,New Zealand Women's,1215,0.071964,4,3,62.36
4,Emilie Haavi,10392,Norway Women's,852,0.066134,5,2,56.88
